<!-- nav -->
[← 8.03 · Python RegExp](08_03_Python_RegExp.ipynb) · [Home](../../README.md) · [8.05 · TypeScript Patterns →](08_05_TypeScript_Patterns.ipynb)

# 8.04 · TypeScript Types

TypeScript writes its types in a grammar of their own, beside its expressions:

```text
let reading: { id: number; celsius?: number } = parse(text) as Reading;
type Readonly<T> = { readonly [K in keyof T]: T[K] };
```

`{ id: number; celsius?: number }`, `Reading` and `{ readonly [K in keyof T]: T[K] }` are types: an object type, a
reference to a named type, a mapped type. `TypeScript Types` is their dialect, its 37 kinds typescript-estree's type
nodes, named and shaped as mbse-programs has them
([CAP-277: TypeScript's types and destructuring patterns as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-277.md)).
Its `Term` lists, beside them, the six TypeScript5 terms a type may hold, a literal's among them, as in the literal
type `"on"`. A type stands where a declaration annotates, mbse-programs', and in TypeScript5's expressions: after `as`
and `satisfies`, in `<T>x`, in a function's annotations and type arguments (tutorial 8, 08.1).

This page builds types from a keyword up, in the builders' fluent calls, prints each as TypeScript writes it, finds
where TypeScript5's terms hold one, shows what a type binds, validates them, and reads the editions that have its kinds.
Like the rest of Part I, it starts from `dialects()`; `typed` holds a type's spec in the dialect's holder,
`S.TypeScript.Types.Expression`, `reference` writes a reference to a named type, and `keyword` a keyword type. No reader
here reads TypeScript's text, in either language: reading TypeScript is mbse-programs' (tutorial 3).

In [1]:
import assert from "node:assert/strict";
import { Standards, Symbolics, Terms } from "@mbse/expressions/Framework";
import * as TypeScript5 from "@mbse/expressions/TypeScript/TypeScript5";
import * as TypeScriptTypes from "@mbse/expressions/TypeScript/Types";
import { dialects } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const TYPES = TypeScriptTypes.Expressions.DIALECT;
const ToText = TypeScriptTypes.Text.ToText;
const reference = (name: string) => (t: any) => t.tstypereference((r: any) => r.typeName((n: any) => n.identifier((i: any) => i.name(name))));
const keyword = (tag: string) => (t: any) => t[tag]((k: any) => k); // a keyword type: `string` is tsstringkeyword

/** The type `spec` writes, held by its dialect's holder, written in a transaction of its own. */
function typed(spec: any): any {
  return store.transaction(() => S.TypeScript.Types.Expression((e: any) => e.root(spec)));
}

/** A type's text, as TypeScript writes it, and its kind. */
function show(held: any): void {
  console.log(ToText(held).padEnd(36), Terms.kindOf(Terms.term(held))[0]);
}

<a id="T08.04.1"></a>
## 08.04.1 · Keywords, references and arrays

The smallest types are keywords, `string`, `number`, `boolean`, `unknown`, each a kind of its own with nothing in it,
`tsstringkeyword` and the rest. A named type, `Reading` or `T`, is a `tstypereference`, its `typeName` an `identifier`,
and, where it is generic, its `typeArguments`, `Array<string>`. `string[]` is a `tsarraytype` of its `elementType`:

In [2]:
const basics = [
  keyword("tsstringkeyword"), keyword("tsnumberkeyword"), keyword("tsunknownkeyword"),
  reference("Reading"),
  (t: any) => t.tstypereference((r: any) => r.typeName((n: any) => n.identifier((i: any) => i.name("Array")))
    .typeArguments((a: any) => a.params([keyword("tsstringkeyword")]))),
  (t: any) => t.tsarraytype((a: any) => a.elementType(keyword("tsstringkeyword"))),
];
for (const spec of basics) show(typed(spec));

string                               tsstringkeyword


number                               tsnumberkeyword


unknown                              tsunknownkeyword


Reading                              tstypereference


Array<string>                        tstypereference


string[]                             tsarraytype


<a id="T08.04.2"></a>
## 08.04.2 · Unions, literal types, object types and function types

Types combine. A union, `string | number`, is a `tsuniontype` of its `types`; a literal type, `"on"`, a `tsliteraltype`
holding TypeScript5's literal; an object type, `{ id: number; celsius?: number }`, a `tstypeliteral` of its `members`,
each a helper only types hold, here a `tspropertysignature`, its `key`, `optional` and annotation; a function type,
`(x: number) => string`, a `tsfunctiontype`, its parameters `TypeScript Patterns`' terms (the next page), its
return type an annotation. `keyof T` is a `tstypeoperator`, `T[K]` a `tsindexedaccesstype`. Each nests the types of
08.04.1, and the printer writes parentheses where TypeScript's precedence needs them:

In [3]:
const annotation = (spec: any) => (a: any) => a.typeAnnotation(spec); // `: spec`, an annotation
const on = (t: any) => t.tsliteraltype((q: any) => q.literal((l: any) => l.literal((r: any) => r.value((v: any) => v.as_scalar("on")).quote('"'))));
const off = (t: any) => t.tsliteraltype((q: any) => q.literal((l: any) => l.literal((r: any) => r.value((v: any) => v.as_scalar("off")).quote('"'))));
const fn = (t: any) => t.tsfunctiontype((f: any) => f
  .params([(p: any) => p.identifier((i: any) => i.name("x").typeAnnotation(annotation(keyword("tsnumberkeyword"))))])
  .returnType(annotation(keyword("tsstringkeyword"))));
const combined = [
  (t: any) => t.tsuniontype((u: any) => u.types([keyword("tsstringkeyword"), keyword("tsnumberkeyword")])),
  (t: any) => t.tsuniontype((u: any) => u.types([on, off])),
  (t: any) => t.tstypeliteral((o: any) => o.members([
    (m: any) => m.tspropertysignature((s: any) => s.key((k: any) => k.identifier((i: any) => i.name("id")))
      .typeAnnotation(annotation(keyword("tsnumberkeyword")))),
    (m: any) => m.tspropertysignature((s: any) => s.key((k: any) => k.identifier((i: any) => i.name("celsius"))).optional(true)
      .typeAnnotation(annotation(keyword("tsnumberkeyword"))))])),
  fn,
  (t: any) => t.tsuniontype((u: any) => u.types([fn, keyword("tsnullkeyword")])),
  (t: any) => t.tstypeoperator((o: any) => o.operator("keyof").typeAnnotation(reference("T"))),
  (t: any) => t.tsindexedaccesstype((a: any) => a.objectType(reference("T")).indexType(reference("K"))),
];
for (const spec of combined) show(typed(spec));

string | number                      tsuniontype


"on" | "off"                         tsuniontype


{ id: number; celsius?: number; }    tstypeliteral


(x: number) => string                tsfunctiontype


((x: number) => string) | null       tsuniontype


keyof T                              tstypeoperator


T[K]                                 tsindexedaccesstype


<a id="T08.04.3"></a>
## 08.04.3 · Where TypeScript5 holds a type

A type is a term of `TypeScript Types`, but TypeScript5's expressions hold one in a few places: `x as Reading` (a
`tsasexpression`), `x satisfies Reading` and `<Reading>x`, each its `typeAnnotation`; an arrow's type parameters,
parameters' annotations and return type; a call's type arguments. TypeScript5 reads `TypeScript Types`' terms, as each
dialect of the family reads the others' (`reads()`), and one printer prints them all (tutorial 8, 08.5), writing a
generic arrow's `<T,>` as a `.tsx` file needs it. Three of TypeScript5's kinds hold a type as a subterm, read from the
store; an annotation, a type parameter and a type argument hold one through TypeScript5's helpers:

In [4]:
const identifier = (name: string) => (e: any) => e.identifier((i: any) => i.name(name));
const hosts: any[] = store.transaction(() => [
  S.TypeScript.TypeScript5.Expression((e: any) => e.root((u: any) => u.tsasexpression((r: any) => r
    .expression(identifier("x")).typeAnnotation(reference("Reading"))))),
  S.TypeScript.TypeScript5.Expression((e: any) => e.root((u: any) => u.tssatisfiesexpression((r: any) => r
    .expression(identifier("x")).typeAnnotation(reference("Reading"))))),
  S.TypeScript.TypeScript5.Expression((e: any) => e.root((u: any) => u.tstypeassertion((r: any) => r
    .typeAnnotation(reference("Reading")).expression(identifier("x"))))),
  S.TypeScript.TypeScript5.Expression((e: any) => e.root((u: any) => u.arrowfunctionexpression((a: any) => a
    .typeParameters((d: any) => d.params([(p: any) => p.name((i: any) => i.name("T"))]))
    .params([(p: any) => p.identifier((i: any) => i.name("v").typeAnnotation(annotation(reference("T"))))])
    .returnType(annotation((t: any) => t.tsarraytype((a: any) => a.elementType(reference("T")))))
    .body((b: any) => b.callexpression((c: any) => c.callee(identifier("wrap"))
      .typeArguments((g: any) => g.params([reference("T")])).arguments([identifier("v")])))))),
]);
for (const host of hosts) console.log(TypeScript5.Text.ToText(host).padEnd(36), Terms.kindOf(Terms.term(host))[0]);
const types = store.registered(TYPES.union);
const held = (store.registered(TypeScript5.Expressions.DIALECT.union).branches as any[]).flatMap((b) => [...b.schema.properties.entries()]
  .filter(([, p]: [string, any]) => p.schema === types || p.schema.element === types).map(([name]: [string, any]) => `${b.name}.${name}`));
console.log("TypeScript5's properties that hold a type:", held.join(", "));
console.log("TypeScript5 reads", TypeScript5.Expressions.DIALECT.reads().map((d: any) => d.name()).join(", "), "| one printer:",
  ToText === TypeScript5.Text.ToText ? "True" : "False");

x as Reading                         tsasexpression


x satisfies Reading                  tssatisfiesexpression


<Reading>x                           tstypeassertion


<T,>(v: T): T[] => wrap<T>(v)        arrowfunctionexpression


TypeScript5's properties that hold a type: tsasexpression.typeAnnotation, tssatisfiesexpression.typeAnnotation, tstypeassertion.typeAnnotation


TypeScript5 reads TypeScript Types, TypeScript Patterns, TypeScript RegExp | one printer: True


<a id="T08.04.4"></a>
## 08.04.4 · What a type binds, and the names it reads

A name in a type, `Reading` or `T`, names a type its program declares, or a value whose type `typeof` reads: something
no term models, so a type's names are references that validation never reports unbound, and that are never free. Three
kinds bind names within a type, and refine the framework's `Let` (tutorial 7): a mapped type binds its key, `K` in
`{ readonly [K in keyof T]: T[K] }`, its constraint Let's `value` and its type Let's `body`; a function type and a
constructor type bind their type parameters and parameters within them. A conditional type binds what its `infer`
declares, `V` in `T extends (infer V)[] ? V : never`, in its true branch. A name a value's place reads is TypeScript5's
`identifier`, which its scope gives, as in a computed key:

In [5]:
const readonly = typed((t: any) => t.tsmappedtype((m: any) => m.readonly("readonly").key((i: any) => i.name("K"))
  .value((v: any) => v.tstypeoperator((o: any) => o.operator("keyof").typeAnnotation(reference("T"))))
  .body((b: any) => b.tsindexedaccesstype((a: any) => a.objectType(reference("T")).indexType(reference("K"))))));
const element = typed((t: any) => t.tsconditionaltype((c: any) => c.checkType(reference("T"))
  .extendsType((e: any) => e.tsarraytype((a: any) => a.elementType(
    (i: any) => i.tsinfertype((f: any) => f.typeParameter((p: any) => p.name((n: any) => n.name("V")))))))
  .trueType(reference("V")).falseType(keyword("tsneverkeyword"))));
const computed = typed((t: any) => t.tstypeliteral((o: any) => o.members([(m: any) => m.tspropertysignature((s: any) => s
  .computed(true).key(identifier("key")).typeAnnotation(annotation(keyword("tsstringkeyword"))))])));
for (const held of [readonly, element, typed(fn), computed]) {
  const kind = Terms.kindOf(Terms.term(held));
  console.log(ToText(held).padEnd(36), kind[0].padEnd(17), "binder:", Terms.isBinder(kind[1].as_schema()) ? "True" : "False", "| free:",
    [...Symbolics.free(held, TYPES)].sort().join(", ") || "none", "|", TYPES.validate(held).join("; ") || "valid");
}
const form = Terms.form(Terms.term(readonly));
console.log("key:", (form.attributes.get("key") as Terms.Part).attributes.get("name"), "| value:", ToText(form.arguments[0]),
  "| body:", ToText(form.arguments[1]));

{ readonly [K in keyof T]: T[K] }    tsmappedtype      binder: True | free: none | valid


T extends (infer V)[] ? V : never    tsconditionaltype binder: False | free: none | valid


(x: number) => string                tsfunctiontype    binder: True | free: none | valid


{ [key]: string; }                   tstypeliteral     binder: False | free: key | members 0 key: identifier 'key' is not bound


key: K | value: keyof T | body: T[K]


<a id="T08.04.5"></a>
## 08.04.5 · Validated

Validation checks each kind as typescript-estree has it, in its own words: a union needs two types, a mapped type its
constraint, a conditional type all four of its parts, and a name, a reference's or a mapped type's key, must be one
TypeScript can spell. A refused type is
kept, and printed as written
([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)):

In [6]:
const refused = [
  (t: any) => t.tsuniontype((u: any) => u.types([keyword("tsstringkeyword")])),
  (t: any) => t.tsmappedtype((m: any) => m.key((i: any) => i.name("K K")).value(reference("T")).body(reference("T"))),
  (t: any) => t.tstypereference((r: any) => r.typeName((n: any) => n.identifier((i: any) => i.name("max-temp")))),
];
for (const spec of refused) {
  const held = typed(spec);
  console.log(ToText(held).padEnd(27), TYPES.validate(held).join("; "));
}
assert.deepEqual(TYPES.validate(typed(refused[0])), ["a tsuniontype needs two types"]);

string                      a tsuniontype needs two types


{ [K K in T]: T }           an identifier's name cannot be 'K K'


max-temp                    typeName: an identifier's name cannot be 'max-temp'


<a id="T08.04.6"></a>
## 08.04.6 · Inferred as any, and not evaluated

A type describes values; it is not one. Inference gives a type the domain `any`, which says nothing, and TypeScript5's
evaluator, which evaluates its literals alone (tutorial 3), refuses each of `TypeScript Types`' kinds by the dialect's
name, as it refuses an `as` by its own:

In [7]:
/** What TypeScript5's evaluator says of `expression`. */
function refusedBy(expression: any): string {
  try {
    return String(TypeScript5.Simplifiers.OfAny(expression).evaluate());
  } catch (error) {
    assert.equal((error as Error).name, "NotImplementedError");
    return (error as Error).message;
  }
}

const union = typed((t: any) => t.tsuniontype((u: any) => u.types([keyword("tsstringkeyword"), keyword("tsnumberkeyword")])));
for (const held of [union, readonly]) {
  console.log(ToText(held).padEnd(36), "inferred:", TYPES.infer(held).name(), "|", refusedBy(held));
}
console.log(TypeScript5.Text.ToText(hosts[0]).padEnd(36), refusedBy(hosts[0]));

string | number                      inferred: any | TypeScript Types tsuniontype is not evaluated


{ readonly [K in keyof T]: T[K] }    inferred: any | TypeScript Types tsmappedtype is not evaluated


x as Reading                         TypeScript5 tsasexpression is not evaluated


<a id="T08.04.7"></a>
## 08.04.7 · Its editions as data

`TypeScript Types` covers the versions of TypeScript 5 that TypeScript5 covers, 5.0 to 5.9, each found by its published
name, Microsoft's, and each with all 37 of its kinds
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md));
the six terms its `Term` lists from TypeScript5 are recorded there:

In [8]:
const dialect = store.Instances.Expressions.TypeScript.Types;
const editions = Standards.standards(dialect);
console.log([...editions.keys()].join(", "));
console.log((Standards.properties(Standards.find(dialect, "TypeScript 5.4")!).get("bodies") as string[])[0], "| kinds in each:",
  [...editions.values()].map((edition) => String(Standards.kinds(edition).length)).join(", "));

TypeScript 5.0, TypeScript 5.1, TypeScript 5.2, TypeScript 5.3, TypeScript 5.4, TypeScript 5.5, TypeScript 5.6, TypeScript 5.7, TypeScript 5.8, TypeScript 5.9


com.microsoft | kinds in each: 37, 37, 37, 37, 37, 37, 37, 37, 37, 37


## What you learned

- **A type is a term of its own dialect**, `TypeScript Types`: keywords, references, arrays, unions, literal, object,
  function, mapped and conditional types, 37 kinds, built from the smallest up.
- **TypeScript5 holds a type** after `as` and `satisfies`, in `<T>x`, and in a function's annotations and type
  arguments, and reads `TypeScript Types`' terms there; one printer writes both.
- **A type's names are references** to what its program declares, never reported unbound; a mapped type, a function
  type and a constructor type bind names within them, and refine `Let`.
- **Validation speaks typescript-estree's rules**; inference gives `any`, and evaluation refuses each kind by name.
- **Its editions are data**: TypeScript 5.0 to 5.9.

Next, `TypeScript Patterns`: the destructuring a parameter or an assignment's target holds.

<!-- nav -->
[← 8.03 · Python RegExp](08_03_Python_RegExp.ipynb) · [Home](../../README.md) · [8.05 · TypeScript Patterns →](08_05_TypeScript_Patterns.ipynb)